In [3]:
import cv2
import numpy as np
import os
from PIL import Image, ImageEnhance, ImageFilter
import random

In [5]:
# ============================================================
# CONFIGURE THESE TWO VARIABLES
# ============================================================
INPUT_IMAGE_PATH = r"pictures_mixed/OneFront.jpg"   # <-- path to the single photo
OUTPUT_FOLDER    = r"augmented/OneFront"             # <-- folder to save augmented images
# ============================================================

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
original = Image.open(INPUT_IMAGE_PATH).convert("RGB")
print(f"Loaded: {INPUT_IMAGE_PATH}  |  Size: {original.size}")

Loaded: pictures_mixed/OneFront.jpg  |  Size: (3060, 4080)


In [6]:
def save(img, tag, counter):
    """Save a PIL image with a sequential name."""
    name = f"{counter:03d}_{tag}.jpg"
    img.save(os.path.join(OUTPUT_FOLDER, name), quality=95)
    return counter + 1


def pil_to_cv(img):
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def cv_to_pil(img):
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [7]:
counter = 1

# --- 0. Original ---
counter = save(original, "original", counter)

# --- 1. Rotations (12 images) ---
for angle in [15, 30, 45, 60, 90, 120, 135, 150, 180, 210, 270, 330]:
    rotated = original.rotate(angle, resample=Image.BICUBIC, expand=True, fillcolor=(0, 0, 0))
    counter = save(rotated, f"rotate_{angle}", counter)

# --- 2. Flips (2 images) ---
counter = save(original.transpose(Image.FLIP_LEFT_RIGHT), "flip_horizontal", counter)
counter = save(original.transpose(Image.FLIP_TOP_BOTTOM), "flip_vertical", counter)

# --- 3. Brightness variations (4 images) ---
for factor in [0.5, 0.75, 1.3, 1.6]:
    bright = ImageEnhance.Brightness(original).enhance(factor)
    counter = save(bright, f"brightness_{factor}", counter)

# --- 4. Contrast variations (4 images) ---
for factor in [0.5, 0.75, 1.4, 1.8]:
    contrast = ImageEnhance.Contrast(original).enhance(factor)
    counter = save(contrast, f"contrast_{factor}", counter)

# --- 5. Saturation / Color variations (4 images) ---
for factor in [0.3, 0.6, 1.4, 1.8]:
    color = ImageEnhance.Color(original).enhance(factor)
    counter = save(color, f"saturation_{factor}", counter)

# --- 6. Sharpness variations (3 images) ---
for factor in [0.3, 2.0, 3.0]:
    sharp = ImageEnhance.Sharpness(original).enhance(factor)
    counter = save(sharp, f"sharpness_{factor}", counter)

# --- 7. Gaussian blur (3 images) ---
for radius in [1, 2, 4]:
    blurred = original.filter(ImageFilter.GaussianBlur(radius=radius))
    counter = save(blurred, f"blur_{radius}", counter)

# --- 8. Gaussian noise (3 images) ---
for sigma in [10, 25, 50]:
    arr = np.array(original, dtype=np.float32)
    noise = np.random.normal(0, sigma, arr.shape)
    noisy = np.clip(arr + noise, 0, 255).astype(np.uint8)
    counter = save(Image.fromarray(noisy), f"noise_{sigma}", counter)

# --- 9. Random crops / zooms (4 images) ---
w, h = original.size
crop_fractions = [0.7, 0.75, 0.8, 0.85]
for i, frac in enumerate(crop_fractions):
    cw, ch = int(w * frac), int(h * frac)
    left = random.randint(0, w - cw)
    top = random.randint(0, h - ch)
    cropped = original.crop((left, top, left + cw, top + ch)).resize((w, h), Image.BICUBIC)
    counter = save(cropped, f"crop_zoom_{i}", counter)

# --- 10. Perspective / affine transforms (4 images) ---
cv_img = pil_to_cv(original)
rows, cols = cv_img.shape[:2]
for i, offset in enumerate([30, 60, 90, 120]):
    pts1 = np.float32([[0, 0], [cols, 0], [0, rows], [cols, rows]])
    pts2 = np.float32([
        [offset, offset],
        [cols - offset, offset // 2],
        [offset // 2, rows - offset],
        [cols - offset, rows - offset]
    ])
    M = cv2.getPerspectiveTransform(pts1, pts2)
    warped = cv2.warpPerspective(cv_img, M, (cols, rows))
    counter = save(cv_to_pil(warped), f"perspective_{i}", counter)

# --- 11. Background color changes (6 images) ---
bg_colors = [
    (0, 0, 0),        # black
    (50, 50, 50),     # dark gray
    (200, 200, 200),  # light gray
    (30, 80, 30),     # dark green
    (139, 90, 43),    # brown / wood
    (70, 70, 120),    # blue-gray
]
gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
for i, color in enumerate(bg_colors):
    bg = np.full_like(cv_img, color[::-1])  # BGR
    result = cv_img.copy()
    result[mask == 255] = bg[mask == 255]
    counter = save(cv_to_pil(result), f"bg_color_{i}", counter)

# --- 12. Combined augmentations (10 images) ---
random.seed(42)
for i in range(10):
    img = original.copy()
    # random rotation
    angle = random.uniform(-25, 25)
    img = img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=(0, 0, 0))
    # random brightness
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.4))
    # random contrast
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
    # random saturation
    img = ImageEnhance.Color(img).enhance(random.uniform(0.5, 1.5))
    # random crop
    frac = random.uniform(0.8, 0.95)
    cw, ch = int(w * frac), int(h * frac)
    left = random.randint(0, w - cw)
    top = random.randint(0, h - ch)
    img = img.crop((left, top, left + cw, top + ch)).resize((w, h), Image.BICUBIC)
    # maybe flip
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    # maybe add noise
    if random.random() > 0.5:
        arr = np.array(img, dtype=np.float32)
        arr += np.random.normal(0, random.uniform(5, 20), arr.shape)
        img = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    counter = save(img, f"combined_{i}", counter)

print(f"\nDone! Generated {counter - 1} augmented images in: {OUTPUT_FOLDER}")

NameError: name 'cv2' is not defined